# CartPole Full SpikeEngine CUDA Scratchpad

This notebook is the `cartpole_lif_reservoir_rl_scratchpad` experiment rebuilt on the real `SpikeEngineCUDA` path. The reservoir is a square-torus spiking engine on GPU with low-rank weights and k2-tree construction enabled. The actor and critic are still simple linear readouts over engine-derived features.

This is intentionally single-environment first. The point is to make the full engine path correct and inspectable before adding a batched multi-engine rollout collector.

In [ ]:
import importlib
import math
import time
from dataclasses import dataclass, replace

import numpy as np
import cupy as cp
import gymnasium as gym
import plotly.graph_objects as go
from tqdm.auto import tqdm

import spike_engine_cuda
import weights_cuda
import topologies

importlib.reload(weights_cuda)
importlib.reload(spike_engine_cuda)

from spike_engine_cuda import SpikeEngineCUDA
from topologies import square_torus

print(f"gymnasium {gym.__version__}; numpy {np.__version__}; cupy {cp.__version__}")

## Configuration

`SpikeEngineReservoirConfig` controls the fixed engine. `RLConfig` controls only the linear actor-critic readout. Inputs are signed, so each CartPole observation channel is split into positive and negative current channels before driving groups of engine neurons.

In [ ]:
@dataclass
class SpikeEngineReservoirConfig:
    side: int = 32
    rank: int = 64
    base_input_dim: int = 6
    input_repeats: int = 8
    resting_mp: float = 0.1
    spike_threshold: float = 1.0
    decay_rate: float = 0.18
    recurrent_scale: float = 1.04
    input_gain: float = 1.35
    spike_tau: float = 12.0
    feature_voltage_scale: float = 1.0
    freeze_learning: bool = True
    weight_init_scale: float = 0.01
    seed: int = 11
    use_k2tree: bool = True
    verify_k2tree: bool = False


@dataclass
class RLConfig:
    env_id: str = "CartPole-v1"
    seed: int = 123
    train_episodes: int = 1500
    eval_episodes: int = 50
    eval_every: int = 100
    max_episode_steps: int | None = None
    gamma: float = 0.97
    actor_lr: float = 8e-2
    critic_lr: float = 2e-2
    reward_scale: float = 0.01
    entropy_beta: float = 0.0
    l2: float = 1e-5
    normalize_advantage: bool = True
    batch_episodes: int = 16
    max_grad_norm: float = 5.0
    advantage_clip: float = 5.0
    max_policy_logit_step: float = 0.08
    entropy_floor: float = 0.45
    entropy_recovery_beta: float = 0.02
    ppo_clip: float = 0.20
    ppo_epochs: int = 2
    target_kl: float = 0.02
    line_search: bool = True
    line_search_backtracks: int = 8
    line_search_decay: float = 0.5
    armijo_c1: float = 1e-4
    hard_start_prob: float = 0.25
    failure_buffer_size: int = 5000
    failure_window: int = 20
    failure_state_jitter: float = 0.01
    failure_weight_bonus: float = 1.0
    failure_weight_window: int = 20
    stochastic_training: bool = True


RESERVOIR_CONFIG = SpikeEngineReservoirConfig()
RL_CONFIG = RLConfig()
print(RESERVOIR_CONFIG)
print(RL_CONFIG)


## Environment Encoding

In [ ]:
def moving_average(values, window: int = 50):
    values = np.asarray(values, dtype=np.float32)
    if len(values) < window:
        return np.full_like(values, np.nan, dtype=np.float32)
    kernel = np.ones(window, dtype=np.float32) / window
    out = np.convolve(values, kernel, mode="valid")
    return np.concatenate([np.full(window - 1, np.nan, dtype=np.float32), out])


def softmax(logits):
    z = np.asarray(logits, dtype=np.float32)
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / np.maximum(1e-8, e.sum(axis=-1, keepdims=True))


def make_env(config: RLConfig, seed: int | None = None):
    if config.max_episode_steps is None:
        env = gym.make(config.env_id)
    else:
        env = gym.make(config.env_id, max_episode_steps=config.max_episode_steps)
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
    return env


def _configured_max_steps(config: RLConfig):
    if config.max_episode_steps is not None:
        return int(config.max_episode_steps)
    return 500


def _step_fraction(step: int, config: RLConfig):
    return min(1.0, float(step) / max(1, _configured_max_steps(config)))


def encode_cartpole_observation(env, observation, previous_action: float = 0.0, step_fraction: float = 0.0):
    obs = np.asarray(observation, dtype=np.float32)
    high = np.asarray(env.observation_space.high, dtype=np.float32)
    scale = np.array([
        high[0] if np.isfinite(high[0]) else 4.8,
        3.0,
        high[2] if np.isfinite(high[2]) else 0.418,
        3.5,
    ], dtype=np.float32)
    physics = np.clip(obs / scale, -2.0, 2.0) / 2.0
    return np.concatenate([
        physics,
        np.asarray([previous_action, step_fraction], dtype=np.float32),
    ]).astype(np.float32)


def signed_channels(input_vector):
    u = np.asarray(input_vector, dtype=np.float32)
    return np.concatenate([np.maximum(u, 0.0), np.maximum(-u, 0.0)]).astype(np.float32)

## SpikeEngine CartPole Reservoir

The wrapper persists one engine state across each episode. At every environment step it injects signed observation current, advances the engine one tick, and exposes membrane plus spike-recency features.

In [ ]:
class SpikeEngineCartPoleReservoir:
    frame_title = "SpikeEngine membrane potential"

    def __init__(self, config: SpikeEngineReservoirConfig):
        self.config = config
        self.n_reservoir = config.side * config.side
        self.encoded_input_dim = 2 * config.base_input_dim
        self.input_neurons = self._make_input_neurons()
        cp.random.seed(config.seed)
        self.engine = SpikeEngineCUDA(
            square_torus(config.side),
            (config.side, config.side),
            rank=config.rank,
            resting_mp=config.resting_mp,
            decay_rate=config.decay_rate,
            weight_initializer=lambda size: cp.random.normal(0.0, config.weight_init_scale, size=size),
            use_k2tree=config.use_k2tree,
            verify_k2tree=config.verify_k2tree,
            verify_progress_every=2000,
        )
        self.engine.SPIKE_THRESHOLD = cp.float32(config.spike_threshold)
        self.engine.set_input_neurons(self.input_neurons)
        target, w_accum, w_instant = self.engine.set_constant_weights_near_bifurcation(
            input_period=1,
            scale=config.recurrent_scale,
            freeze_learning=config.freeze_learning,
        )
        self.weight_summary = {"target": target, "w_accum": w_accum, "w_instant": w_instant}
        self.reset()

    @property
    def n_features(self):
        return 2 * self.n_reservoir + self.encoded_input_dim + 1

    def _make_input_neurons(self):
        cfg = self.config
        rng = np.random.default_rng(cfg.seed)
        total = self.encoded_input_dim * cfg.input_repeats
        if total > cfg.side * cfg.side:
            raise ValueError("input_repeats too high for reservoir size")
        ids = rng.choice(cfg.side * cfg.side, size=total, replace=False)
        return cp.asarray(ids.reshape(self.encoded_input_dim, cfg.input_repeats).ravel(), dtype=cp.int32)

    def reset(self):
        self.engine.reset_state()
        self.tick = 2
        self.last_encoded_input = np.zeros(self.encoded_input_dim, dtype=np.float32)

    def apply_runtime_controls(
        self,
        *,
        input_gain: float | None = None,
        recurrent_scale: float | None = None,
        decay_rate: float | None = None,
        spike_threshold: float | None = None,
        resting_mp: float | None = None,
        spike_tau: float | None = None,
        feature_voltage_scale: float | None = None,
        reset_state: bool = False,
    ):
        cfg = self.config
        if input_gain is not None:
            cfg.input_gain = float(input_gain)
        if decay_rate is not None:
            cfg.decay_rate = float(decay_rate)
            self.engine.DECAY_RATE = cp.float32(cfg.decay_rate)
        if spike_threshold is not None:
            cfg.spike_threshold = float(spike_threshold)
            self.engine.SPIKE_THRESHOLD = cp.float32(cfg.spike_threshold)
        if resting_mp is not None:
            old_rest = float(self.engine.RESTING_MP)
            cfg.resting_mp = float(resting_mp)
            self.engine.RESTING_MP = cp.float32(cfg.resting_mp)
            self.engine.membrane_potentials += cp.float32(cfg.resting_mp - old_rest)
        if spike_tau is not None:
            cfg.spike_tau = float(spike_tau)
        if feature_voltage_scale is not None:
            cfg.feature_voltage_scale = float(feature_voltage_scale)
        if recurrent_scale is not None:
            cfg.recurrent_scale = float(recurrent_scale)
            target, w_accum, w_instant = self.engine.set_constant_weights_near_bifurcation(
                input_period=1,
                scale=cfg.recurrent_scale,
                freeze_learning=cfg.freeze_learning,
            )
            self.weight_summary = {"target": target, "w_accum": w_accum, "w_instant": w_instant}
        if reset_state:
            self.reset()
        return self.weight_summary

    def step(self, input_vector):
        encoded = signed_channels(input_vector)
        self.last_encoded_input = encoded
        values = np.repeat(encoded, self.config.input_repeats).astype(np.float32) * self.config.input_gain
        self.engine.advance_static_input(values, self.tick, self.input_neurons, full_decay=False)
        self.tick += 1

    def features(self, input_vector=None):
        self.engine._decay_all(self.tick)
        mp = cp.asnumpy(self.engine.membrane_potentials).astype(np.float32)
        last = cp.asnumpy(self.engine.last_spiked).astype(np.int32)
        age = np.maximum(0, self.tick - last).astype(np.float32)
        trace = np.exp(-age / max(1e-6, self.config.spike_tau)).astype(np.float32)
        trace[last <= 0] = 0.0
        voltage = ((mp - self.config.resting_mp) / max(1e-6, self.config.feature_voltage_scale)).astype(np.float32)
        encoded = self.last_encoded_input if input_vector is None else signed_channels(input_vector)
        return np.concatenate([trace, voltage, encoded, np.ones(1, dtype=np.float32)]).astype(np.float32)

    def frame(self):
        self.engine._decay_all(self.tick)
        return cp.asnumpy(self.engine.membrane_potentials).reshape(self.config.side, self.config.side)

    def stats(self):
        frame = self.frame()
        last = cp.asnumpy(self.engine.last_spiked)
        recent = np.mean((self.tick - last) <= 1)
        return {"mp_mean": float(frame.mean()), "mp_std": float(frame.std()), "recent_spike_fraction": float(recent)}


reservoir = SpikeEngineCartPoleReservoir(RESERVOIR_CONFIG)
print("reservoir neurons:", reservoir.n_reservoir, "features:", reservoir.n_features)
print("input neurons:", len(reservoir.input_neurons), "k2tree constructed:", reservoir.engine.weights.k2tree is not None)
print("weight summary:", reservoir.weight_summary)


## Linear Actor-Critic Readout

This keeps the trained parameter surface small: a linear softmax actor and a linear value function over engine features. Updates are batched over complete episodes.

In [ ]:
class ReservoirActorCritic:
    def __init__(self, n_features: int, config: RLConfig):
        self.n_features = int(n_features)
        self.config = replace(config)
        self.rng = np.random.default_rng(config.seed + 2)
        self.actor_W = np.empty((self.n_features, 2), dtype=np.float32)
        self.value_W = np.empty(self.n_features, dtype=np.float32)
        self.reset_weights(seed=config.seed + 2)

    @property
    def n_trained_parameters(self):
        return int(self.actor_W.size + self.value_W.size)

    def _reset_metrics(self):
        self.updates = 0
        self.last_metrics = {}
        self.last_value_loss = None
        self.last_policy_loss = None
        self.last_policy_entropy = None
        self.last_episode_advantage_mean = None
        self.last_batch_episodes = 0
        self.last_batch_steps = 0
        self.last_actor_grad_norm = None
        self.last_value_grad_norm = None
        self.last_policy_logit_step = None
        self.last_policy_step_scale = 1.0
        self.last_approx_kl = None
        self.last_clip_fraction = None
        self.last_ppo_epochs = 0
        self.last_line_search_steps = 0
        self.last_line_search_scale = 1.0
        self.last_line_search_accepted = True
        self.last_hard_start_count = int(getattr(self, "last_hard_start_count", 0))
        self.last_failure_buffer_size = len(getattr(self, "failure_state_buffer", []))

    def reset_weights(self, seed: int | None = None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self.failure_state_buffer = []
        self.last_hard_start_count = 0
        self.actor_W = self.rng.normal(0.0, 1e-3, size=(self.n_features, 2)).astype(np.float32)
        self.value_W = np.zeros(self.n_features, dtype=np.float32)
        self._reset_metrics()

    def logits(self, features):
        F = np.asarray(features, dtype=np.float32)
        if F.ndim == 1:
            F = F[None, :]
        return F @ self.actor_W

    def probabilities(self, features):
        return softmax(self.logits(features))

    def value(self, features):
        F = np.asarray(features, dtype=np.float32)
        return F @ self.value_W

    def choose_action(self, features, stochastic: bool = True):
        probs = self.probabilities(features)[0]
        if stochastic:
            action = int(self.rng.choice(2, p=probs))
        else:
            action = int(np.argmax(probs))
        return action, probs

    def choose_actions(self, features, stochastic: bool = True):
        probs = self.probabilities(features)
        if stochastic:
            draws = self.rng.random(probs.shape[0])
            actions = (draws > probs[:, 0]).astype(np.int64)
        else:
            actions = np.argmax(probs, axis=1).astype(np.int64)
        return actions, probs

    def _discounted_returns(self, rewards):
        rewards = np.asarray(rewards, dtype=np.float32)
        returns = np.zeros_like(rewards, dtype=np.float32)
        running_return = 0.0
        for idx in range(len(rewards) - 1, -1, -1):
            running_return = rewards[idx] + self.config.gamma * running_return
            returns[idx] = running_return
        return returns

    def _clip_gradient(self, grad):
        max_norm = float(getattr(self.config, "max_grad_norm", 0.0))
        norm = float(np.linalg.norm(grad))
        if not np.isfinite(norm):
            return np.zeros_like(grad, dtype=np.float32), norm
        if max_norm > 0.0 and norm > max_norm:
            grad = grad * (max_norm / (norm + 1e-8))
        return grad, norm

    def _policy_loss(self, actor_W, F, actions, old_log_probs, old_probs, actor_advantages, sample_weights, entropy_beta):
        n = int(actions.shape[0])
        logits = F @ actor_W
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(logits)
        probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
        if not np.all(np.isfinite(probs)):
            return np.inf, np.nan, np.inf, np.inf, np.nan

        row_idx = np.arange(n)
        log_probs = np.log(probs[row_idx, actions] + 1e-8)
        ratio = np.exp(np.clip(log_probs - old_log_probs, -20.0, 20.0))
        ppo_clip = float(getattr(self.config, "ppo_clip", 0.0))
        if ppo_clip > 0.0:
            clipped_ratio = np.clip(ratio, 1.0 - ppo_clip, 1.0 + ppo_clip)
            surrogate = np.minimum(ratio * actor_advantages, clipped_ratio * actor_advantages)
            clip_fraction = float(np.mean(sample_weights * (np.abs(ratio - 1.0) > ppo_clip)))
        else:
            surrogate = ratio * actor_advantages
            clip_fraction = 0.0

        entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
        entropy_mean = float(np.mean(sample_weights * entropy))
        full_kl = float(np.mean(sample_weights * np.sum(old_probs * (np.log(old_probs + 1e-8) - np.log(probs + 1e-8)), axis=1)))
        sampled_kl = float(np.mean(sample_weights * (old_log_probs - log_probs)))
        loss = -float(np.mean(sample_weights * surrogate)) - float(entropy_beta) * entropy_mean
        loss += 0.5 * float(self.config.l2) * float(np.sum(actor_W * actor_W))
        return loss, entropy_mean, max(0.0, full_kl), sampled_kl, clip_fraction

    def update_episodes(self, trajectories):
        feature_batches, action_batches, return_batches, weight_batches = [], [], [], []
        for trajectory in trajectories:
            if trajectory is None or len(trajectory.get("rewards", [])) == 0:
                continue
            rewards = trajectory["rewards"]
            feature_batches.append(np.asarray(trajectory["features"], dtype=np.float32))
            action_batches.append(np.asarray(trajectory["actions"], dtype=np.int64))
            return_batches.append(self._discounted_returns(rewards))
            weights = np.asarray(trajectory.get("sample_weights", np.ones(len(rewards), dtype=np.float32)), dtype=np.float32)
            if len(weights) != len(rewards):
                weights = np.ones(len(rewards), dtype=np.float32)
            weight_batches.append(weights)
        if not feature_batches:
            return {"skipped": True, "reason": "empty batch"}

        F = np.vstack(feature_batches).astype(np.float32)
        actions = np.concatenate(action_batches).astype(np.int64)
        returns = np.concatenate(return_batches).astype(np.float32)
        sample_weights = np.concatenate(weight_batches).astype(np.float32)
        sample_weights = np.clip(sample_weights, 0.0, 20.0)
        if float(sample_weights.sum()) <= 0.0:
            sample_weights = np.ones_like(sample_weights, dtype=np.float32)
        sample_weights *= len(sample_weights) / max(1e-8, float(sample_weights.sum()))
        if not (np.all(np.isfinite(F)) and np.all(np.isfinite(returns)) and np.all(np.isfinite(sample_weights))):
            return {"skipped": True, "reason": "non-finite features or returns"}

        old_probs = self.probabilities(F).astype(np.float32)
        if not np.all(np.isfinite(old_probs)):
            return {"skipped": True, "reason": "non-finite old policy probabilities"}
        n = int(actions.shape[0])
        row_idx = np.arange(n)
        old_log_probs = np.log(old_probs[row_idx, actions] + 1e-8).astype(np.float32)

        values_before = F @ self.value_W
        advantages = returns - values_before
        actor_advantages = advantages.copy()
        if self.config.normalize_advantage and n > 1:
            actor_advantages = (advantages - advantages.mean()) / max(advantages.std(), 0.05)
        advantage_clip = float(getattr(self.config, "advantage_clip", 0.0))
        if advantage_clip > 0.0:
            actor_advantages = np.clip(actor_advantages, -advantage_clip, advantage_clip)

        selected = np.zeros((n, 2), dtype=np.float32)
        selected[row_idx, actions] = 1.0

        ppo_epochs = max(1, int(getattr(self.config, "ppo_epochs", 1)))
        ppo_clip = float(getattr(self.config, "ppo_clip", 0.0))
        target_kl = float(getattr(self.config, "target_kl", 0.0))
        use_line_search = bool(getattr(self.config, "line_search", True))
        backtracks = max(1, int(getattr(self.config, "line_search_backtracks", 8)))
        decay = float(np.clip(getattr(self.config, "line_search_decay", 0.5), 0.05, 0.95))
        armijo_c1 = float(getattr(self.config, "armijo_c1", 1e-4))
        last_metrics = None

        for epoch in range(ppo_epochs):
            probs = self.probabilities(F)
            if not np.all(np.isfinite(probs)):
                return {"skipped": True, "reason": "non-finite policy probabilities"}
            log_probs = np.log(probs[row_idx, actions] + 1e-8)
            ratio = np.exp(np.clip(log_probs - old_log_probs, -20.0, 20.0))

            if ppo_clip > 0.0:
                active = (
                    ((actor_advantages >= 0.0) & (ratio <= 1.0 + ppo_clip))
                    | ((actor_advantages < 0.0) & (ratio >= 1.0 - ppo_clip))
                ).astype(np.float32)
            else:
                active = np.ones_like(actor_advantages, dtype=np.float32)

            entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
            entropy_mean = float(np.mean(sample_weights * entropy))
            entropy_beta = float(self.config.entropy_beta)
            entropy_floor = float(getattr(self.config, "entropy_floor", 0.0))
            if entropy_floor > 0.0 and entropy_mean < entropy_floor:
                recovery = float(getattr(self.config, "entropy_recovery_beta", 0.0))
                entropy_beta = max(entropy_beta, recovery * (entropy_floor - entropy_mean) / max(entropy_floor, 1e-6))

            policy_weights = actor_advantages * ratio * active * sample_weights
            grad_logits = (probs - selected) * policy_weights[:, None]
            if entropy_beta != 0.0:
                entropy_grad = probs * (np.log(probs + 1e-8) + entropy[:, None])
                grad_logits += entropy_beta * entropy_grad

            values = F @ self.value_W
            grad_actor = F.T @ grad_logits / n
            grad_actor += self.config.l2 * self.actor_W
            grad_value = F.T @ ((values - returns) * sample_weights) / n
            grad_value += self.config.l2 * self.value_W
            grad_actor, actor_grad_norm = self._clip_gradient(grad_actor)
            grad_value, value_grad_norm = self._clip_gradient(grad_value)
            if not (np.all(np.isfinite(grad_actor)) and np.all(np.isfinite(grad_value))):
                return {"skipped": True, "reason": "non-finite gradients"}

            current_loss, _, current_kl, _, _ = self._policy_loss(
                self.actor_W, F, actions, old_log_probs, old_probs, actor_advantages, sample_weights, entropy_beta
            )
            if target_kl > 0.0 and current_kl > target_kl:
                break

            actor_step = -self.config.actor_lr * grad_actor.astype(np.float32)
            logit_step = F @ actor_step
            logit_step_rms = float(np.sqrt(np.mean(logit_step * logit_step)))
            max_logit_step = float(getattr(self.config, "max_policy_logit_step", 0.0))
            logit_step_scale = 1.0
            if max_logit_step > 0.0 and logit_step_rms > max_logit_step:
                logit_step_scale = max_logit_step / (logit_step_rms + 1e-8)
                actor_step *= logit_step_scale

            grad_dot_step = float(np.sum(grad_actor * actor_step))
            accepted = not use_line_search
            line_scale = 1.0
            line_steps = 0
            candidate_loss = current_loss
            candidate_entropy = entropy_mean
            candidate_kl = current_kl
            candidate_sampled_kl = 0.0
            candidate_clip_fraction = 0.0

            if use_line_search:
                accepted = False
                if grad_dot_step < 0.0 and np.isfinite(current_loss):
                    for line_steps in range(1, backtracks + 1):
                        candidate_W = self.actor_W + line_scale * actor_step
                        candidate_loss, candidate_entropy, candidate_kl, candidate_sampled_kl, candidate_clip_fraction = self._policy_loss(
                            candidate_W, F, actions, old_log_probs, old_probs, actor_advantages, sample_weights, entropy_beta
                        )
                        sufficient_decrease = candidate_loss <= current_loss + armijo_c1 * line_scale * grad_dot_step
                        kl_ok = (target_kl <= 0.0) or (candidate_kl <= target_kl)
                        if np.isfinite(candidate_loss) and sufficient_decrease and kl_ok:
                            accepted = True
                            break
                        line_scale *= decay
                if not accepted:
                    line_scale = 0.0
                    actor_step = np.zeros_like(actor_step, dtype=np.float32)
                    candidate_loss, candidate_entropy, candidate_kl, candidate_sampled_kl, candidate_clip_fraction = self._policy_loss(
                        self.actor_W, F, actions, old_log_probs, old_probs, actor_advantages, sample_weights, entropy_beta
                    )
            else:
                candidate_W = self.actor_W + actor_step
                candidate_loss, candidate_entropy, candidate_kl, candidate_sampled_kl, candidate_clip_fraction = self._policy_loss(
                    candidate_W, F, actions, old_log_probs, old_probs, actor_advantages, sample_weights, entropy_beta
                )

            self.actor_W += line_scale * actor_step
            self.value_W -= self.config.critic_lr * grad_value.astype(np.float32)
            self.actor_W = np.clip(self.actor_W, -10.0, 10.0)
            self.value_W = np.clip(self.value_W, -100.0, 100.0)

            last_metrics = {
                "value_loss": float(np.mean(sample_weights * (values - returns) ** 2)),
                "policy_loss": float(candidate_loss),
                "entropy": float(candidate_entropy),
                "advantage_mean": float(np.mean(advantages)),
                "actor_grad_norm": float(actor_grad_norm),
                "value_grad_norm": float(value_grad_norm),
                "policy_logit_step": logit_step_rms,
                "policy_step_scale": float(logit_step_scale * line_scale),
                "approx_kl": float(candidate_kl),
                "sampled_kl": float(candidate_sampled_kl),
                "clip_fraction": float(candidate_clip_fraction),
                "line_search_steps": int(line_steps),
                "line_search_scale": float(line_scale),
                "line_search_accepted": bool(accepted),
                "ppo_epochs": epoch + 1,
                "batch_episodes": len(feature_batches),
                "batch_steps": int(n),
            }
            if use_line_search and not accepted:
                break
            if target_kl > 0.0 and candidate_kl > target_kl:
                break

        if last_metrics is None:
            return {"skipped": True, "reason": "no policy epoch ran"}

        self.updates += 1
        self.last_metrics = last_metrics
        self.last_value_loss = last_metrics["value_loss"]
        self.last_policy_loss = last_metrics["policy_loss"]
        self.last_policy_entropy = last_metrics["entropy"]
        self.last_episode_advantage_mean = last_metrics["advantage_mean"]
        self.last_batch_episodes = last_metrics["batch_episodes"]
        self.last_batch_steps = last_metrics["batch_steps"]
        self.last_actor_grad_norm = last_metrics["actor_grad_norm"]
        self.last_value_grad_norm = last_metrics["value_grad_norm"]
        self.last_policy_logit_step = last_metrics["policy_logit_step"]
        self.last_policy_step_scale = last_metrics["policy_step_scale"]
        self.last_approx_kl = last_metrics["approx_kl"]
        self.last_clip_fraction = last_metrics["clip_fraction"]
        self.last_line_search_steps = last_metrics["line_search_steps"]
        self.last_line_search_scale = last_metrics["line_search_scale"]
        self.last_line_search_accepted = last_metrics["line_search_accepted"]
        self.last_ppo_epochs = last_metrics["ppo_epochs"]
        return last_metrics

    def update_episode(self, features, actions, rewards):
        return self.update_episodes([
            {"features": features, "actions": actions, "rewards": rewards}
        ])


## Rollout, Training, Evaluation

In [ ]:
def make_system(reservoir_config: SpikeEngineReservoirConfig = RESERVOIR_CONFIG, rl_config: RLConfig = RL_CONFIG):
    env = make_env(rl_config, seed=rl_config.seed)
    reservoir = SpikeEngineCartPoleReservoir(reservoir_config)
    agent = ReservoirActorCritic(reservoir.n_features, rl_config)
    return env, reservoir, agent


def _failure_buffer(agent):
    if not hasattr(agent, "failure_state_buffer"):
        agent.failure_state_buffer = []
    return agent.failure_state_buffer


def _failure_buffer_size(config):
    return max(0, int(getattr(config, "failure_buffer_size", 0)))


def _store_failure_states(agent, states, config):
    buffer_size = _failure_buffer_size(config)
    window = max(0, int(getattr(config, "failure_window", 0)))
    if buffer_size <= 0 or window <= 0 or not states:
        return
    buffer = _failure_buffer(agent)
    for state in states[-window:]:
        buffer.append(np.asarray(state, dtype=np.float32).copy())
    if len(buffer) > buffer_size:
        del buffer[: len(buffer) - buffer_size]
    agent.last_failure_buffer_size = len(buffer)


def _sample_failure_start(agent, env, config):
    buffer = _failure_buffer(agent)
    if not buffer:
        return None
    state = np.asarray(buffer[int(agent.rng.integers(len(buffer)))], dtype=np.float32).copy()
    jitter = float(getattr(config, "failure_state_jitter", 0.0))
    if jitter > 0.0:
        state += agent.rng.normal(0.0, jitter, size=state.shape).astype(np.float32)
    x_limit = float(env.unwrapped.x_threshold) * 0.995
    theta_limit = float(env.unwrapped.theta_threshold_radians) * 0.995
    state[0] = np.clip(state[0], -x_limit, x_limit)
    state[1] = np.clip(state[1], -5.0, 5.0)
    state[2] = np.clip(state[2], -theta_limit, theta_limit)
    state[3] = np.clip(state[3], -5.0, 5.0)
    return state.astype(np.float32)


def _reset_env(env, agent, config, seed: int | None = None, train: bool = False):
    observation, _ = env.reset(seed=seed)
    prob = float(getattr(config, "hard_start_prob", 0.0))
    if train and prob > 0.0 and _failure_buffer(agent) and agent.rng.random() < prob:
        state = _sample_failure_start(agent, env, config)
        if state is not None:
            env.unwrapped.state = state.copy()
            observation = state
            agent.last_hard_start_count = int(getattr(agent, "last_hard_start_count", 0)) + 1
    agent.last_failure_buffer_size = len(_failure_buffer(agent))
    return np.asarray(observation, dtype=np.float32)


def _failure_sample_weights(num_steps: int, did_fail: bool, config):
    num_steps = int(num_steps)
    weights = np.ones(num_steps, dtype=np.float32)
    bonus = float(getattr(config, "failure_weight_bonus", 0.0))
    window = max(0, int(getattr(config, "failure_weight_window", getattr(config, "failure_window", 0))))
    if did_fail and bonus > 0.0 and window > 0 and num_steps > 0:
        start = max(0, num_steps - window)
        ramp = np.linspace(1.0 / max(1, num_steps - start), 1.0, num_steps - start, dtype=np.float32)
        weights[start:] += bonus * ramp
        weights *= num_steps / max(1e-8, float(weights.sum()))
    return weights.astype(np.float32)


def run_episode(
    env,
    reservoir,
    agent: ReservoirActorCritic,
    config: RLConfig,
    train: bool = True,
    stochastic: bool = True,
    seed: int | None = None,
    render_callback=None,
    delay_s: float = 0.0,
    update: bool = False,
    return_trajectory: bool = True,
    stop_callback=None,
):
    observation = _reset_env(env, agent, config, seed=seed, train=train)
    reservoir.reset()
    previous_action = 0.0
    features, actions, rewards, log_probs = [], [], [], []
    recent_states = []
    total_reward = 0.0
    last_probs = np.full(2, 0.5, dtype=np.float32)
    terminated_flag = False
    truncated_flag = False
    stopped = False
    max_steps = _configured_max_steps(config)
    step = 0

    while max_steps is None or step < max_steps:
        if stop_callback is not None and stop_callback():
            stopped = True
            break
        recent_states.append(np.asarray(observation, dtype=np.float32).copy())
        input_vector = encode_cartpole_observation(env, observation, previous_action, _step_fraction(step, config))
        reservoir.step(input_vector)
        feature = reservoir.features(input_vector)
        use_stochastic = bool(stochastic and (not train or config.stochastic_training))
        action, probs = agent.choose_action(feature, stochastic=use_stochastic)
        last_probs = probs
        next_observation, reward, terminated, truncated, _ = env.step(action)
        terminated_flag = bool(terminated)
        truncated_flag = bool(truncated)
        done = bool(terminated_flag or truncated_flag)
        scaled_reward = float(reward) * config.reward_scale

        features.append(feature)
        actions.append(action)
        rewards.append(scaled_reward)
        log_probs.append(float(np.log(probs[action] + 1e-8)))
        total_reward += float(reward)

        if render_callback is not None:
            render_callback(
                observation=next_observation,
                action=action,
                probs=probs,
                step=step + 1,
                total_reward=total_reward,
                done=done,
            )
            if delay_s > 0.0:
                time.sleep(delay_s)
        if stop_callback is not None and stop_callback():
            stopped = True
            break

        previous_action = 1.0 if action == 1 else -1.0
        observation = next_observation
        step += 1
        if done:
            break

    did_fail = bool(train and terminated_flag)
    if did_fail:
        _store_failure_states(agent, recent_states, config)
    sample_weights = _failure_sample_weights(len(rewards), did_fail, config)
    trajectory = {"features": features, "actions": actions, "rewards": rewards, "log_probs": log_probs, "sample_weights": sample_weights}
    metrics = agent.update_episodes([trajectory]) if train and update else None
    result = {
        "return": float(total_reward),
        "steps": len(rewards),
        "terminated": terminated_flag,
        "truncated": truncated_flag,
        "stopped": stopped,
        "last_probs": last_probs,
        "metrics": metrics,
    }
    if return_trajectory:
        result["trajectory"] = trajectory
    return result


def evaluate_agent(env, reservoir, agent, config: RLConfig, episodes: int | None = None, seed_offset: int = 10_000, stop_callback=None):
    episodes = config.eval_episodes if episodes is None else int(episodes)
    returns = []
    for idx in range(episodes):
        if stop_callback is not None and stop_callback():
            break
        result = run_episode(
            env,
            reservoir,
            agent,
            config,
            train=False,
            stochastic=False,
            seed=config.seed * 100_000 + seed_offset + idx,
            update=False,
            return_trajectory=False,
            stop_callback=stop_callback,
        )
        if result.get("stopped") and result["steps"] == 0:
            break
        returns.append(result["return"])
    return np.asarray(returns, dtype=np.float32)


def train_agent(env, reservoir, agent, config: RLConfig, episodes: int | None = None, eval_every: int | None = None, progress: bool = True, stop_callback=None):
    episodes = config.train_episodes if episodes is None else int(episodes)
    eval_every = config.eval_every if eval_every is None else int(eval_every or 0)
    agent.config = replace(config)
    returns = []
    eval_history = []
    pending = []
    iterator = range(episodes)
    if progress:
        iterator = tqdm(iterator, total=episodes, desc="SpikeEngine actor-critic")
    for episode in iterator:
        if stop_callback is not None and stop_callback():
            break
        result = run_episode(
            env,
            reservoir,
            agent,
            config,
            train=True,
            stochastic=True,
            seed=config.seed * 100_000 + episode,
            update=False,
            return_trajectory=True,
            stop_callback=stop_callback,
        )
        if result.get("stopped") and result["steps"] == 0:
            break
        returns.append(result["return"])
        pending.append(result["trajectory"])
        if len(pending) >= config.batch_episodes or episode == episodes - 1 or result.get("stopped"):
            agent.update_episodes(pending)
            pending.clear()
        if result.get("stopped"):
            break
        if eval_every and (episode + 1) % eval_every == 0:
            eval_returns = evaluate_agent(
                env,
                reservoir,
                agent,
                config,
                episodes=max(10, min(30, config.eval_episodes)),
                seed_offset=20_000 + episode * 100,
                stop_callback=stop_callback,
            )
            if len(eval_returns):
                eval_history.append((episode + 1, float(eval_returns.mean()), float(eval_returns.min()), float(eval_returns.max())))
            if progress:
                iterator.set_postfix(train_ma50=f"{np.mean(returns[-50:]):.1f}", eval=f"{eval_returns.mean():.1f}" if len(eval_returns) else "n/a", updates=agent.updates)
    return {
        "returns": np.asarray(returns, dtype=np.float32),
        "eval_history": eval_history,
        "reservoir_stats": reservoir.stats(),
        "failure_buffer_size": len(_failure_buffer(agent)),
        "hard_starts": int(getattr(agent, "last_hard_start_count", 0)),
    }


## Default Training Run

This is intentionally the same workflow as the CartPole scratchpads: build the system, train the readout, evaluate greedily, then plot returns.

In [ ]:
env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)
print(f"feature dimension: {reservoir.n_features}")
print(f"trained parameters: {agent.n_trained_parameters}")
print("reservoir stats before training:", reservoir.stats())

start_time = time.perf_counter()
history = train_agent(env, reservoir, agent, RL_CONFIG, eval_every=100)
train_seconds = time.perf_counter() - start_time
eval_returns = evaluate_agent(env, reservoir, agent, RL_CONFIG)

returns = history["returns"]
print(f"training time: {train_seconds:.2f}s")
print(f"train last 50: {returns[-50:].mean():.1f}")
print(f"greedy eval mean/min/max over {len(eval_returns)} episodes: {eval_returns.mean():.1f} / {eval_returns.min():.1f} / {eval_returns.max():.1f}")
print("last update metrics:", agent.last_metrics)
print("reservoir stats after training:", history["reservoir_stats"])

## Training Plot

In [ ]:
def plot_training_history(history, eval_returns=None):
    returns = np.asarray(history["returns"], dtype=np.float32)
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=returns, mode="lines", name="train return", line=dict(width=1, color="#9ecae1")))
    fig.add_trace(go.Scatter(y=moving_average(returns, 50), mode="lines", name="train MA50", line=dict(width=3, color="#1f77b4")))
    if history.get("eval_history"):
        episodes, means, mins, maxs = zip(*history["eval_history"])
        fig.add_trace(go.Scatter(x=episodes, y=means, mode="lines+markers", name="greedy eval mean", line=dict(width=3, color="#f58518")))
        fig.add_trace(go.Scatter(x=episodes, y=maxs, mode="markers", name="greedy eval max", marker=dict(color="#54a24b", size=7)))
    if eval_returns is not None:
        fig.add_hline(y=float(np.mean(eval_returns)), line_dash="dash", annotation_text=f"final eval mean {float(np.mean(eval_returns)):.1f}")
    fig.update_layout(width=950, height=420, title="CartPole SpikeEngine actor-critic training", xaxis_title="episode", yaxis_title="return")
    fig.show()
    return fig

training_fig = plot_training_history(history, eval_returns)

## CartPole Drawing Helpers

In [ ]:
def cartpole_traces(observation, env):
    x, _, theta, _ = np.asarray(observation, dtype=np.float32)
    cart_y = 0.0
    cart_w = 0.42
    cart_h = 0.22
    pole_len = 1.0
    px = x + pole_len * math.sin(theta)
    py = cart_y + pole_len * math.cos(theta)
    return [
        go.Scatter(x=[-2.6, 2.6], y=[-0.14, -0.14], mode="lines", line=dict(color="#777", width=2), showlegend=False),
        go.Scatter(x=[x - cart_w, x + cart_w, x + cart_w, x - cart_w, x - cart_w], y=[cart_y - cart_h, cart_y - cart_h, cart_y + cart_h, cart_y + cart_h, cart_y - cart_h], mode="lines", fill="toself", line=dict(color="#4c78a8", width=2), fillcolor="#9ecae1", showlegend=False),
        go.Scatter(x=[x, px], y=[cart_y + cart_h, py], mode="lines+markers", line=dict(color="#f58518", width=6), marker=dict(size=[5, 9]), showlegend=False),
    ]


def make_cartpole_figure(env):
    fig = go.FigureWidget(data=cartpole_traces(np.zeros(4, dtype=np.float32), env))
    fig.update_layout(width=620, height=320, margin=dict(l=20, r=20, t=35, b=20), xaxis=dict(range=[-2.6, 2.6], zeroline=False), yaxis=dict(range=[-0.35, 1.25], scaleanchor="x", scaleratio=1, zeroline=False), title=dict(text="CartPole rollout", x=0.5, font=dict(size=14)))
    return fig


def update_cartpole_figure(fig, observation, env):
    traces = cartpole_traces(observation, env)
    with fig.batch_update():
        for idx, trace in enumerate(traces):
            fig.data[idx].x = trace.x
            fig.data[idx].y = trace.y

## Live Workbench

The workbench keeps the same trained engine and readout alive. Use it for more training, greedy evaluation, or a rollout with the reservoir frame and policy bars.

In [ ]:
try:
    import threading
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    widgets = None
    print(f"ipywidgets unavailable: {exc}")


class CartPoleSpikeEngineWorkbench:
    def __init__(self, env, reservoir, agent, config, history=None):
        if widgets is None:
            raise RuntimeError("ipywidgets is required for the workbench")
        self.env = env
        self.reservoir = reservoir
        self.agent = agent
        self.config = replace(config)
        self.agent.config = replace(self.config)
        base_history = history or {"returns": [], "eval_history": []}
        self.history = {
            "returns": list(np.asarray(base_history.get("returns", []), dtype=float)),
            "eval_history": list(base_history.get("eval_history", [])),
        }
        self.stop_flag = False
        self._thread = None
        self.world_fig = make_cartpole_figure(env)
        self.policy_fig = go.FigureWidget(data=[go.Bar(x=["left", "right"], y=[0.5, 0.5], marker_color=["#4c78a8", "#4c78a8"])])
        self.policy_fig.update_layout(width=300, height=230, margin=dict(l=35, r=10, t=30, b=35), yaxis=dict(range=[0, 1], fixedrange=True), title=dict(text="policy", x=0.5, font=dict(size=13)))
        self.reservoir_fig = go.FigureWidget(data=[go.Heatmap(z=self.reservoir.frame(), colorscale="Viridis", showscale=False)])
        self.reservoir_fig.update_layout(width=300, height=300, margin=dict(l=5, r=5, t=30, b=5), title=dict(text=self.reservoir.frame_title, x=0.5, font=dict(size=13)), xaxis=dict(visible=False), yaxis=dict(visible=False, autorange="reversed"))
        self.return_fig = go.FigureWidget()
        self.return_fig.update_layout(width=950, height=320, margin=dict(l=45, r=20, t=35, b=40), xaxis_title="episode", yaxis_title="return", title=dict(text="live training returns", x=0.5, font=dict(size=14)))
        self.train_button = widgets.Button(description="Train", icon="graduation-cap", button_style="success")
        self.eval_button = widgets.Button(description="Evaluate", icon="bar-chart", button_style="info")
        self.rollout_button = widgets.Button(description="Rollout", icon="play", button_style="primary")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="warning")
        self.reset_button = widgets.Button(description="Reset Agent", icon="refresh", button_style="")
        self.reset_reservoir_button = widgets.Button(description="Reset Reservoir", icon="refresh", button_style="")
        self.apply_dynamics_button = widgets.Button(description="Apply Dynamics", icon="sliders", button_style="warning")
        slider_style = {"description_width": "initial"}
        rcfg = self.reservoir.config
        self.train_episodes = widgets.IntSlider(value=100, min=1, max=5000, step=1, description="episodes", continuous_update=False)
        self.eval_episodes = widgets.IntSlider(value=20, min=1, max=200, step=1, description="eval n", continuous_update=False)
        self.eval_every = widgets.IntSlider(value=0, min=0, max=1000, step=10, description="eval every", continuous_update=False, style=slider_style)
        self.plot_every = widgets.IntSlider(value=1, min=1, max=100, step=1, description="plot every", continuous_update=False, style=slider_style)
        self.batch_episodes = widgets.IntSlider(value=self.config.batch_episodes, min=1, max=128, step=1, description="batch eps", continuous_update=False, style=slider_style)
        self.delay_ms = widgets.IntSlider(value=20, min=0, max=250, step=5, description="delay ms", continuous_update=False, style=slider_style)
        self.stochastic_rollout = widgets.Checkbox(value=False, description="stochastic rollout")
        self.reset_on_dynamics = widgets.Checkbox(value=True, description="reset on dynamics apply", style=slider_style)
        self.actor_lr = widgets.FloatLogSlider(value=self.config.actor_lr, base=10, min=-4, max=0, step=0.1, description="actor lr", continuous_update=False, style=slider_style)
        self.critic_lr = widgets.FloatLogSlider(value=self.config.critic_lr, base=10, min=-5, max=0, step=0.1, description="critic lr", continuous_update=False, style=slider_style)
        self.gamma = widgets.FloatSlider(value=self.config.gamma, min=0.90, max=0.999, step=0.001, readout_format=".3f", description="gamma", continuous_update=False)
        self.reward_scale = widgets.FloatLogSlider(value=self.config.reward_scale, base=10, min=-4, max=0, step=0.1, description="reward scale", continuous_update=False, style=slider_style)
        self.entropy_beta = widgets.FloatLogSlider(value=max(1e-6, self.config.entropy_beta if self.config.entropy_beta > 0 else 1e-6), base=10, min=-6, max=-1, step=0.25, description="entropy beta", continuous_update=False, style=slider_style)
        self.use_entropy_beta = widgets.Checkbox(value=self.config.entropy_beta > 0.0, description="base entropy")
        self.max_grad_norm = widgets.FloatSlider(value=self.config.max_grad_norm, min=0.5, max=20.0, step=0.5, readout_format=".1f", description="grad clip", continuous_update=False, style=slider_style)
        self.policy_step = widgets.FloatSlider(value=float(getattr(self.config, "max_policy_logit_step", 0.08)), min=0.01, max=0.25, step=0.01, readout_format=".2f", description="policy step", continuous_update=False, style=slider_style)
        self.entropy_floor = widgets.FloatSlider(value=float(getattr(self.config, "entropy_floor", 0.45)), min=0.0, max=0.68, step=0.01, readout_format=".2f", description="entropy min", continuous_update=False, style=slider_style)
        self.ppo_clip = widgets.FloatSlider(value=float(getattr(self.config, "ppo_clip", 0.20)), min=0.05, max=0.75, step=0.01, readout_format=".2f", description="ppo clip", continuous_update=False, style=slider_style)
        self.target_kl = widgets.FloatSlider(value=float(getattr(self.config, "target_kl", 0.02)), min=0.0, max=0.10, step=0.005, readout_format=".3f", description="target kl", continuous_update=False, style=slider_style)
        self.ppo_epochs = widgets.IntSlider(value=int(getattr(self.config, "ppo_epochs", 2)), min=1, max=32, step=1, description="ppo epochs", continuous_update=False, style=slider_style)
        self.line_search = widgets.Checkbox(value=bool(getattr(self.config, "line_search", True)), description="line search")
        self.backtracks = widgets.IntSlider(value=int(getattr(self.config, "line_search_backtracks", 8)), min=1, max=16, step=1, description="backtracks", continuous_update=False, style=slider_style)
        self.hard_start_prob = widgets.FloatSlider(value=float(getattr(self.config, "hard_start_prob", 0.25)), min=0.0, max=1.0, step=0.05, readout_format=".2f", description="hard starts", continuous_update=False, style=slider_style)
        self.failure_weight_bonus = widgets.FloatSlider(value=float(getattr(self.config, "failure_weight_bonus", 1.0)), min=0.0, max=4.0, step=0.1, readout_format=".1f", description="fail weight", continuous_update=False, style=slider_style)
        self.failure_window = widgets.IntSlider(value=int(getattr(self.config, "failure_window", 20)), min=1, max=100, step=1, description="fail window", continuous_update=False, style=slider_style)
        self.input_gain = widgets.FloatSlider(value=rcfg.input_gain, min=0.0, max=6.0, step=0.05, description="input gain", continuous_update=False, style=slider_style)
        self.recurrent_scale = widgets.FloatSlider(value=rcfg.recurrent_scale, min=0.0, max=2.5, step=0.01, readout_format=".2f", description="recurrent scale", continuous_update=False, style=slider_style)
        self.decay_rate = widgets.FloatSlider(value=rcfg.decay_rate, min=0.0, max=0.75, step=0.01, readout_format=".2f", description="decay", continuous_update=False, style=slider_style)
        self.spike_threshold = widgets.FloatSlider(value=rcfg.spike_threshold, min=0.05, max=3.0, step=0.05, readout_format=".2f", description="threshold", continuous_update=False, style=slider_style)
        self.resting_mp = widgets.FloatSlider(value=rcfg.resting_mp, min=-0.5, max=0.95, step=0.01, readout_format=".2f", description="resting MP", continuous_update=False, style=slider_style)
        self.spike_tau = widgets.FloatSlider(value=rcfg.spike_tau, min=1.0, max=80.0, step=1.0, readout_format=".0f", description="spike trace tau", continuous_update=False, style=slider_style)
        self.feature_voltage_scale = widgets.FloatLogSlider(value=rcfg.feature_voltage_scale, base=10, min=-2, max=1, step=0.1, description="voltage scale", continuous_update=False, style=slider_style)
        self.reservoir_zmax = widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05, description="mp zmax", continuous_update=False, style=slider_style)
        self.status = widgets.HTML(value="ready")
        self.train_button.on_click(lambda _: self.train_async())
        self.eval_button.on_click(lambda _: self.evaluate_async())
        self.rollout_button.on_click(lambda _: self.rollout_async())
        self.stop_button.on_click(lambda _: self.stop())
        self.reset_button.on_click(lambda _: self.reset_agent())
        self.reset_reservoir_button.on_click(lambda _: self.reset_reservoir())
        self.apply_dynamics_button.on_click(lambda _: self.apply_dynamics())
        self._refresh_return_figure()

    def _sync_config(self, reset_state: bool = False):
        self.config.batch_episodes = int(self.batch_episodes.value)
        self.config.actor_lr = float(self.actor_lr.value)
        self.config.critic_lr = float(self.critic_lr.value)
        self.config.gamma = float(self.gamma.value)
        self.config.reward_scale = float(self.reward_scale.value)
        self.config.entropy_beta = float(self.entropy_beta.value) if self.use_entropy_beta.value else 0.0
        self.config.max_grad_norm = float(self.max_grad_norm.value)
        self.config.max_policy_logit_step = float(self.policy_step.value)
        self.config.entropy_floor = float(self.entropy_floor.value)
        self.config.ppo_clip = float(self.ppo_clip.value)
        self.config.target_kl = float(self.target_kl.value)
        self.config.ppo_epochs = int(self.ppo_epochs.value)
        self.config.line_search = bool(self.line_search.value)
        self.config.line_search_backtracks = int(self.backtracks.value)
        self.config.hard_start_prob = float(self.hard_start_prob.value)
        self.config.failure_weight_bonus = float(self.failure_weight_bonus.value)
        self.config.failure_window = int(self.failure_window.value)
        self.config.failure_weight_window = int(self.failure_window.value)
        self.agent.config = replace(self.config)
        self.reservoir.apply_runtime_controls(
            input_gain=float(self.input_gain.value),
            recurrent_scale=float(self.recurrent_scale.value),
            decay_rate=float(self.decay_rate.value),
            spike_threshold=float(self.spike_threshold.value),
            resting_mp=float(self.resting_mp.value),
            spike_tau=float(self.spike_tau.value),
            feature_voltage_scale=float(self.feature_voltage_scale.value),
            reset_state=reset_state,
        )

    def _reservoir_status(self):
        stats = self.reservoir.stats()
        return (
            f"recent={stats['recent_spike_fraction']:.3f} mp_mean={stats['mp_mean']:.3f} "
            f"mp_max={float(self.reservoir.frame().max()):.3f} target_w={self.reservoir.weight_summary['target']:.4f}"
        )

    def _metric_status(self):
        grad = self.agent.last_actor_grad_norm
        step = self.agent.last_policy_logit_step
        kl = self.agent.last_approx_kl
        entropy = self.agent.last_policy_entropy
        grad_text = "n/a" if grad is None else f"{grad:.2g}"
        step_text = "n/a" if step is None else f"{step:.2g}"
        kl_text = "n/a" if kl is None else f"{kl:.3f}"
        entropy_text = "n/a" if entropy is None else f"{entropy:.3f}"
        return (
            f"updates={self.agent.updates} grad={grad_text} step={step_text} kl={kl_text} entropy={entropy_text} "
            f"scale={self.agent.last_policy_step_scale:.2f} bt={self.agent.last_line_search_steps} "
            f"failbuf={len(_failure_buffer(self.agent))} hard={self.agent.last_hard_start_count}"
        )

    def _refresh_return_figure(self):
        returns = np.asarray(self.history.get("returns", []), dtype=np.float32)
        with self.return_fig.batch_update():
            self.return_fig.data = []
            if len(returns):
                self.return_fig.add_trace(go.Scatter(y=returns, mode="lines", name="return", line=dict(color="#9ecae1", width=1)))
                self.return_fig.add_trace(go.Scatter(y=moving_average(returns, 50), mode="lines", name="MA50", line=dict(color="#1f77b4", width=3)))
            if self.history.get("eval_history"):
                episodes, means, mins, maxs = zip(*self.history["eval_history"])
                self.return_fig.add_trace(go.Scatter(x=episodes, y=means, mode="lines+markers", name="eval mean", line=dict(color="#f58518", width=3)))

    def _draw_policy(self, probs):
        pred = int(np.argmax(probs))
        colors = ["#4c78a8", "#4c78a8"]
        colors[pred] = "#f58518"
        with self.policy_fig.batch_update():
            self.policy_fig.data[0].y = probs
            self.policy_fig.data[0].marker.color = colors
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].z = self.reservoir.frame()
            self.reservoir_fig.data[0].zmax = float(self.reservoir_zmax.value)

    def _render_callback(self, observation, action, probs, step, total_reward, done):
        update_cartpole_figure(self.world_fig, observation, self.env)
        self._draw_policy(probs)
        self.status.value = f"step={step} action={action} return={total_reward:.0f} {self._reservoir_status()}"

    def apply_dynamics(self):
        self._sync_config(reset_state=bool(self.reset_on_dynamics.value))
        self._draw_policy([0.5, 0.5])
        self.status.value = f"dynamics applied {self._reservoir_status()}"

    def train(self, episodes: int | None = None):
        self._sync_config()
        self.stop_flag = False
        episodes = int(self.train_episodes.value if episodes is None else episodes)
        batch_episodes = max(1, int(self.batch_episodes.value))
        plot_every = max(1, int(self.plot_every.value))
        eval_every = max(0, int(self.eval_every.value))
        start_count = len(self.history["returns"])
        pending = []
        self.status.value = f"training {episodes} episodes..."
        for idx in range(episodes):
            if self.stop_flag:
                break
            result = run_episode(
                self.env,
                self.reservoir,
                self.agent,
                self.config,
                train=True,
                stochastic=True,
                seed=self.config.seed * 100_000 + start_count + idx,
                update=False,
                return_trajectory=True,
                stop_callback=lambda: self.stop_flag,
            )
            if result.get("stopped") and result["steps"] == 0:
                break
            self.history["returns"].append(float(result["return"]))
            if len(result["trajectory"].get("rewards", [])):
                pending.append(result["trajectory"])
            should_update = len(pending) >= batch_episodes or idx == episodes - 1 or result.get("stopped")
            if should_update and pending:
                self.agent.update_episodes(pending)
                pending.clear()
            trained_count = len(self.history["returns"])
            if eval_every and trained_count > 0 and trained_count % eval_every == 0:
                eval_returns = evaluate_agent(self.env, self.reservoir, self.agent, self.config, episodes=int(self.eval_episodes.value), seed_offset=60_000 + trained_count * 100, stop_callback=lambda: self.stop_flag)
                if len(eval_returns):
                    self.history["eval_history"].append((trained_count, float(eval_returns.mean()), float(eval_returns.min()), float(eval_returns.max())))
            if (idx + 1) % plot_every == 0 or should_update:
                returns = np.asarray(self.history["returns"], dtype=np.float32)
                ma50 = float(np.mean(returns[-50:])) if len(returns) else np.nan
                self._refresh_return_figure()
                self.status.value = f"trained={trained_count} last={result['return']:.0f} MA50={ma50:.1f} {self._metric_status()} {self._reservoir_status()}"
            if result.get("stopped"):
                break
        if pending:
            self.agent.update_episodes(pending)
        self._refresh_return_figure()
        returns = np.asarray(self.history["returns"], dtype=np.float32)
        ma50 = float(np.mean(returns[-50:])) if len(returns) else np.nan
        state = "stopped" if self.stop_flag else "trained"
        self.status.value = f"{state}={len(returns)} MA50={ma50:.1f} {self._metric_status()} {self._reservoir_status()}"
        self.stop_flag = False

    def evaluate(self):
        self._sync_config()
        self.status.value = "evaluating..."
        returns = evaluate_agent(self.env, self.reservoir, self.agent, self.config, episodes=int(self.eval_episodes.value), stop_callback=lambda: self.stop_flag)
        if len(returns):
            self.history["eval_history"].append((len(self.history["returns"]), float(returns.mean()), float(returns.min()), float(returns.max())))
            self._refresh_return_figure()
            self.status.value = f"eval mean/min/max={returns.mean():.1f}/{returns.min():.0f}/{returns.max():.0f} {self._metric_status()} {self._reservoir_status()}"
        else:
            self.status.value = "eval stopped"

    def rollout(self):
        self._sync_config()
        self.stop_flag = False
        result = run_episode(
            self.env,
            self.reservoir,
            self.agent,
            self.config,
            train=False,
            stochastic=bool(self.stochastic_rollout.value),
            update=False,
            return_trajectory=False,
            render_callback=self._render_callback,
            delay_s=float(self.delay_ms.value) / 1000.0,
            stop_callback=lambda: self.stop_flag,
        )
        self.status.value = f"rollout return={result['return']:.0f} steps={result['steps']} {self._reservoir_status()}"
        self.stop_flag = False

    def _start_thread(self, target):
        if self._thread is not None and self._thread.is_alive():
            self.stop()
        self.stop_flag = False
        self._thread = threading.Thread(target=target, daemon=True)
        self._thread.start()

    def train_async(self):
        self._start_thread(self.train)

    def evaluate_async(self):
        self._start_thread(self.evaluate)

    def rollout_async(self):
        self._start_thread(self.rollout)

    def stop(self):
        self.stop_flag = True
        if self._thread is not None and self._thread.is_alive() and threading.current_thread() is not self._thread:
            self._thread.join(timeout=2)
        self.status.value = "stop requested"

    def reset_agent(self):
        self.stop()
        self.agent.reset_weights(self.config.seed + self.agent.updates + 10)
        self.history = {"returns": [], "eval_history": []}
        self._refresh_return_figure()
        self.status.value = "agent reset"

    def reset_reservoir(self):
        self.reservoir.reset()
        self._draw_policy([0.5, 0.5])
        self.status.value = f"reservoir reset {self._reservoir_status()}"

    def display(self):
        display(widgets.VBox([
            widgets.HBox([self.train_button, self.eval_button, self.rollout_button, self.stop_button, self.reset_button, self.reset_reservoir_button, self.apply_dynamics_button]),
            widgets.HBox([self.train_episodes, self.eval_episodes, self.eval_every, self.plot_every, self.batch_episodes, self.delay_ms, self.stochastic_rollout, self.reset_on_dynamics]),
            widgets.HBox([self.actor_lr, self.critic_lr, self.gamma, self.reward_scale, self.entropy_beta, self.use_entropy_beta, self.max_grad_norm]),
            widgets.HBox([self.policy_step, self.entropy_floor, self.ppo_clip, self.target_kl, self.ppo_epochs, self.line_search, self.backtracks]),
            widgets.HBox([self.hard_start_prob, self.failure_weight_bonus, self.failure_window]),
            widgets.HBox([self.input_gain, self.recurrent_scale, self.decay_rate, self.spike_threshold]),
            widgets.HBox([self.resting_mp, self.spike_tau, self.feature_voltage_scale, self.reservoir_zmax]),
            self.status,
            self.return_fig,
            widgets.HBox([self.world_fig, widgets.VBox([self.policy_fig, self.reservoir_fig])]),
        ]))


try:
    cartpole_workbench.stop()
except NameError:
    pass

cartpole_workbench = CartPoleSpikeEngineWorkbench(env, reservoir, agent, RL_CONFIG, history)
cartpole_workbench.display()


## SpikeEngine Reservoir Parameter Optimization

This section restores the Optuna/TPE workflow from the LIF CartPole scratchpad for the full `SpikeEngineCUDA` reservoir. It searches engine/reservoir parameters and actor-critic training hyperparameters, then exposes the same leaderboard and apply-result helpers for longer confirmation runs.

In [ ]:
try:
    import warnings
    import optuna
    warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError as exc:
    optuna = None
    print(f"Optuna is not installed in this environment: {exc}")


SPIKE_ENGINE_PARAMETER_BOUNDS = {
    "side": (24, 32, 40, 48),
    "rank": (16, 32, 64, 96),
    "input_repeats": (2, 4, 8, 12, 16),
    "decay_rate": (0.05, 0.45),
    "resting_mp": (-0.15, 0.45),
    "spike_threshold": (0.45, 2.0),
    "recurrent_scale": (0.85, 1.25),
    "input_gain": (0.25, 4.0),
    "spike_tau": (3.0, 40.0),
    "feature_voltage_scale": (0.25, 5.0),
    "weight_init_scale": (1e-4, 5e-2),
}

TRAINING_PARAMETER_BOUNDS = {
    "gamma": (0.90, 0.995),
    "actor_lr": (1e-4, 2e-1),
    "critic_lr": (1e-4, 1e-1),
    "reward_scale": (1e-3, 5e-2),
    "entropy_beta": (0.0, 1e-2),
    "max_grad_norm": (0.5, 20.0),
    "batch_episodes": (4, 8, 16, 32, 64),
    "max_policy_logit_step": (0.01, 0.25),
    "entropy_floor": (0.0, 0.68),
    "ppo_clip": (0.10, 0.75),
    "ppo_epochs": (1, 2, 4, 8, 16, 32),
    "target_kl": (0.0, 0.10),
    "line_search_backtracks": (4, 6, 8, 10, 12, 16),
    "hard_start_prob": (0.0, 0.33),
    "failure_weight_bonus": (0.0, 3.0),
    "failure_window": (5, 10, 20, 40, 80),
}


def spike_engine_config_from_params(params, base_config: SpikeEngineReservoirConfig = RESERVOIR_CONFIG, seed: int | None = None):
    cfg_seed = base_config.seed if seed is None else int(seed)
    return replace(
        base_config,
        seed=cfg_seed,
        side=int(params.get("side", base_config.side)),
        rank=int(params.get("rank", base_config.rank)),
        input_repeats=int(params.get("input_repeats", base_config.input_repeats)),
        decay_rate=float(params.get("decay_rate", base_config.decay_rate)),
        resting_mp=float(params.get("resting_mp", base_config.resting_mp)),
        spike_threshold=float(params.get("spike_threshold", base_config.spike_threshold)),
        recurrent_scale=float(params.get("recurrent_scale", base_config.recurrent_scale)),
        input_gain=float(params.get("input_gain", base_config.input_gain)),
        spike_tau=float(params.get("spike_tau", base_config.spike_tau)),
        feature_voltage_scale=float(params.get("feature_voltage_scale", base_config.feature_voltage_scale)),
        weight_init_scale=float(params.get("weight_init_scale", base_config.weight_init_scale)),
    )


def rl_config_from_params(params, base_config: RLConfig = RL_CONFIG, seed: int | None = None):
    cfg_seed = base_config.seed if seed is None else int(seed)
    return replace(
        base_config,
        seed=cfg_seed,
        gamma=float(params.get("gamma", base_config.gamma)),
        actor_lr=float(params.get("actor_lr", base_config.actor_lr)),
        critic_lr=float(params.get("critic_lr", base_config.critic_lr)),
        reward_scale=float(params.get("reward_scale", base_config.reward_scale)),
        entropy_beta=float(params.get("entropy_beta", base_config.entropy_beta)),
        max_grad_norm=float(params.get("max_grad_norm", base_config.max_grad_norm)),
        batch_episodes=int(params.get("batch_episodes", base_config.batch_episodes)),
        max_policy_logit_step=float(params.get("max_policy_logit_step", base_config.max_policy_logit_step)),
        entropy_floor=float(params.get("entropy_floor", base_config.entropy_floor)),
        ppo_clip=float(params.get("ppo_clip", base_config.ppo_clip)),
        ppo_epochs=int(params.get("ppo_epochs", base_config.ppo_epochs)),
        target_kl=float(params.get("target_kl", base_config.target_kl)),
        line_search_backtracks=int(params.get("line_search_backtracks", base_config.line_search_backtracks)),
        hard_start_prob=float(params.get("hard_start_prob", base_config.hard_start_prob)),
        failure_weight_bonus=float(params.get("failure_weight_bonus", base_config.failure_weight_bonus)),
        failure_window=int(params.get("failure_window", base_config.failure_window)),
        failure_weight_window=int(params.get("failure_window", base_config.failure_weight_window)),
    )


def suggest_spike_engine_parameters(trial):
    return {
        "side": trial.suggest_categorical("side", list(SPIKE_ENGINE_PARAMETER_BOUNDS["side"])),
        "rank": trial.suggest_categorical("rank", list(SPIKE_ENGINE_PARAMETER_BOUNDS["rank"])),
        "input_repeats": trial.suggest_categorical("input_repeats", list(SPIKE_ENGINE_PARAMETER_BOUNDS["input_repeats"])),
        "decay_rate": trial.suggest_float("decay_rate", *SPIKE_ENGINE_PARAMETER_BOUNDS["decay_rate"]),
        "resting_mp": trial.suggest_float("resting_mp", *SPIKE_ENGINE_PARAMETER_BOUNDS["resting_mp"]),
        "spike_threshold": trial.suggest_float("spike_threshold", *SPIKE_ENGINE_PARAMETER_BOUNDS["spike_threshold"]),
        "recurrent_scale": trial.suggest_float("recurrent_scale", *SPIKE_ENGINE_PARAMETER_BOUNDS["recurrent_scale"]),
        "input_gain": trial.suggest_float("input_gain", *SPIKE_ENGINE_PARAMETER_BOUNDS["input_gain"], log=True),
        "spike_tau": trial.suggest_float("spike_tau", *SPIKE_ENGINE_PARAMETER_BOUNDS["spike_tau"], log=True),
        "feature_voltage_scale": trial.suggest_float("feature_voltage_scale", *SPIKE_ENGINE_PARAMETER_BOUNDS["feature_voltage_scale"], log=True),
        "weight_init_scale": trial.suggest_float("weight_init_scale", *SPIKE_ENGINE_PARAMETER_BOUNDS["weight_init_scale"], log=True),
    }


def suggest_training_parameters(trial):
    return {
        "gamma": trial.suggest_float("gamma", *TRAINING_PARAMETER_BOUNDS["gamma"]),
        "actor_lr": trial.suggest_float("actor_lr", *TRAINING_PARAMETER_BOUNDS["actor_lr"], log=True),
        "critic_lr": trial.suggest_float("critic_lr", *TRAINING_PARAMETER_BOUNDS["critic_lr"], log=True),
        "reward_scale": trial.suggest_float("reward_scale", *TRAINING_PARAMETER_BOUNDS["reward_scale"], log=True),
        "entropy_beta": trial.suggest_float("entropy_beta", *TRAINING_PARAMETER_BOUNDS["entropy_beta"]),
        "max_grad_norm": trial.suggest_float("max_grad_norm", *TRAINING_PARAMETER_BOUNDS["max_grad_norm"], log=True),
        "batch_episodes": trial.suggest_categorical("batch_episodes", list(TRAINING_PARAMETER_BOUNDS["batch_episodes"])),
        "max_policy_logit_step": trial.suggest_float("max_policy_logit_step", *TRAINING_PARAMETER_BOUNDS["max_policy_logit_step"]),
        "entropy_floor": trial.suggest_float("entropy_floor", *TRAINING_PARAMETER_BOUNDS["entropy_floor"]),
        "ppo_clip": trial.suggest_float("ppo_clip", *TRAINING_PARAMETER_BOUNDS["ppo_clip"]),
        "ppo_epochs": trial.suggest_categorical("ppo_epochs", list(TRAINING_PARAMETER_BOUNDS["ppo_epochs"])),
        "target_kl": trial.suggest_float("target_kl", *TRAINING_PARAMETER_BOUNDS["target_kl"]),
        "line_search_backtracks": trial.suggest_categorical("line_search_backtracks", list(TRAINING_PARAMETER_BOUNDS["line_search_backtracks"])),
        "hard_start_prob": trial.suggest_float("hard_start_prob", *TRAINING_PARAMETER_BOUNDS["hard_start_prob"]),
        "failure_weight_bonus": trial.suggest_float("failure_weight_bonus", *TRAINING_PARAMETER_BOUNDS["failure_weight_bonus"]),
        "failure_window": trial.suggest_categorical("failure_window", list(TRAINING_PARAMETER_BOUNDS["failure_window"])),
    }


def evaluate_spike_engine_candidate(reservoir_config, rl_config, train_episodes: int, eval_episodes: int, trial=None):
    trial_env, trial_reservoir, trial_agent = make_system(reservoir_config, rl_config)
    train_config = replace(rl_config, train_episodes=int(train_episodes), eval_episodes=int(eval_episodes))
    history = train_agent(
        trial_env,
        trial_reservoir,
        trial_agent,
        train_config,
        episodes=int(train_episodes),
        eval_every=0,
        progress=False,
    )
    eval_returns = evaluate_agent(
        trial_env,
        trial_reservoir,
        trial_agent,
        train_config,
        episodes=int(eval_episodes),
        seed_offset=90_000 + train_config.seed,
    )
    returns = np.asarray(history["returns"], dtype=np.float32)
    eval_mean = float(eval_returns.mean()) if len(eval_returns) else 0.0
    eval_std = float(eval_returns.std()) if len(eval_returns) else 0.0
    train_last50 = float(returns[-50:].mean()) if len(returns) else np.nan
    robust_score = eval_mean - 0.25 * eval_std + 0.10 * train_last50
    if trial is not None:
        trial.report(float(robust_score), step=int(train_episodes))
        if trial.should_prune():
            raise optuna.TrialPruned()
    return {
        "score": float(robust_score),
        "eval_mean": eval_mean,
        "eval_std": eval_std,
        "eval_min": float(eval_returns.min()) if len(eval_returns) else 0.0,
        "eval_max": float(eval_returns.max()) if len(eval_returns) else 0.0,
        "train_last50": train_last50,
        "updates": int(trial_agent.updates),
        "history": history,
        "eval_returns": eval_returns,
        "reservoir_stats": trial_reservoir.stats(),
        "failure_buffer_size": int(history.get("failure_buffer_size", 0)),
        "hard_starts": int(history.get("hard_starts", 0)),
    }


def run_spike_engine_optuna_search(
    n_trials: int = 30,
    train_episodes: int = 300,
    eval_episodes: int = 20,
    base_reservoir_config: SpikeEngineReservoirConfig = RESERVOIR_CONFIG,
    base_rl_config: RLConfig = RL_CONFIG,
    seed: int = 123,
    optimize_training: bool = True,
    study_name: str | None = None,
    storage: str | None = None,
    load_if_exists: bool = True,
    progress: bool = True,
):
    if optuna is None:
        raise RuntimeError("Install optuna to run the parameter search.")
    startup = max(5, min(12, int(n_trials) // 4 if int(n_trials) >= 8 else int(n_trials)))
    sampler = optuna.samplers.TPESampler(seed=int(seed), n_startup_trials=startup, multivariate=True, group=True)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=max(3, min(startup, int(n_trials))), n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner, study_name=study_name, storage=storage, load_if_exists=load_if_exists)
    pbar = tqdm(total=int(n_trials), desc="Optuna SpikeEngine TPE", disable=not progress)

    def objective(trial):
        params = suggest_spike_engine_parameters(trial)
        if optimize_training:
            params.update(suggest_training_parameters(trial))
        trial_seed = int(seed) * 10_000 + trial.number
        reservoir_config = spike_engine_config_from_params(params, base_config=base_reservoir_config, seed=trial_seed)
        rl_config = rl_config_from_params(params, base_config=base_rl_config, seed=trial_seed)
        result = evaluate_spike_engine_candidate(reservoir_config, rl_config, int(train_episodes), int(eval_episodes), trial=trial)
        trial.set_user_attr("config_seed", trial_seed)
        trial.set_user_attr("train_episodes", int(train_episodes))
        trial.set_user_attr("eval_episodes", int(eval_episodes))
        for key in ["eval_mean", "eval_std", "eval_min", "eval_max", "train_last50", "updates", "reservoir_stats", "failure_buffer_size", "hard_starts"]:
            trial.set_user_attr(key, result[key])
        return result["score"]

    def update_progress(study, trial):
        pbar.update(1)
        try:
            pbar.set_postfix(best=f"{study.best_value:.1f}", last=trial.state.name)
        except ValueError:
            pbar.set_postfix(best="n/a", last=trial.state.name)

    study.optimize(objective, n_trials=int(n_trials), callbacks=[update_progress], show_progress_bar=False, gc_after_trial=True)
    pbar.close()
    return study, spike_engine_optuna_results(study, base_reservoir_config, base_rl_config)


def spike_engine_optuna_results(study, base_reservoir_config: SpikeEngineReservoirConfig = RESERVOIR_CONFIG, base_rl_config: RLConfig = RL_CONFIG):
    results = []
    for trial in study.trials:
        if trial.state.name != "COMPLETE" or trial.value is None:
            continue
        trial_seed = int(trial.user_attrs.get("config_seed", base_reservoir_config.seed))
        reservoir_config = spike_engine_config_from_params(trial.params, base_config=base_reservoir_config, seed=trial_seed)
        rl_config = rl_config_from_params(trial.params, base_config=base_rl_config, seed=trial_seed)
        results.append({
            "trial": int(trial.number),
            "score": float(trial.value),
            "eval_mean": float(trial.user_attrs.get("eval_mean", trial.value)),
            "eval_std": float(trial.user_attrs.get("eval_std", np.nan)),
            "eval_min": float(trial.user_attrs.get("eval_min", np.nan)),
            "eval_max": float(trial.user_attrs.get("eval_max", np.nan)),
            "train_last50": float(trial.user_attrs.get("train_last50", np.nan)),
            "updates": int(trial.user_attrs.get("updates", 0)),
            "reservoir_config": reservoir_config,
            "rl_config": rl_config,
            "params": dict(trial.params),
            "reservoir_stats": trial.user_attrs.get("reservoir_stats", None),
            "failure_buffer_size": int(trial.user_attrs.get("failure_buffer_size", 0)),
            "hard_starts": int(trial.user_attrs.get("hard_starts", 0)),
        })
    results.sort(key=lambda item: item["score"], reverse=True)
    return results


def summarize_search_result(result):
    rcfg = result["reservoir_config"]
    lcfg = result["rl_config"]
    return {
        "trial": result["trial"],
        "score": result["score"],
        "eval_mean": result["eval_mean"],
        "eval_std": result["eval_std"],
        "eval_min": result["eval_min"],
        "eval_max": result["eval_max"],
        "train_last50": result["train_last50"],
        "updates": result["updates"],
        "side": rcfg.side,
        "rank": rcfg.rank,
        "input_repeats": rcfg.input_repeats,
        "decay_rate": rcfg.decay_rate,
        "resting_mp": rcfg.resting_mp,
        "spike_threshold": rcfg.spike_threshold,
        "recurrent_scale": rcfg.recurrent_scale,
        "input_gain": rcfg.input_gain,
        "spike_tau": rcfg.spike_tau,
        "feature_voltage_scale": rcfg.feature_voltage_scale,
        "weight_init_scale": rcfg.weight_init_scale,
        "gamma": lcfg.gamma,
        "actor_lr": lcfg.actor_lr,
        "critic_lr": lcfg.critic_lr,
        "reward_scale": lcfg.reward_scale,
        "entropy_beta": lcfg.entropy_beta,
        "batch_episodes": lcfg.batch_episodes,
        "max_grad_norm": lcfg.max_grad_norm,
        "max_policy_logit_step": lcfg.max_policy_logit_step,
        "entropy_floor": lcfg.entropy_floor,
        "ppo_clip": lcfg.ppo_clip,
        "ppo_epochs": lcfg.ppo_epochs,
        "target_kl": lcfg.target_kl,
        "line_search_backtracks": lcfg.line_search_backtracks,
        "hard_start_prob": lcfg.hard_start_prob,
        "failure_weight_bonus": lcfg.failure_weight_bonus,
        "failure_window": lcfg.failure_window,
        "failure_buffer_size": result.get("failure_buffer_size", 0),
        "hard_starts": result.get("hard_starts", 0),
        "seed": rcfg.seed,
    }


def print_spike_engine_optimizer_leaderboard(results, top_k: int = 10):
    if not results:
        print("no completed optimization results")
        return []
    rows = [summarize_search_result(result) for result in results[:top_k]]
    for row in rows:
        print(
            f"trial={row['trial']:03d} score={row['score']:.1f} eval_mean={row['eval_mean']:.1f} "
            f"eval={row['eval_min']:.0f}-{row['eval_max']:.0f} train50={row['train_last50']:.1f} updates={row['updates']} | "
            f"side={row['side']} rank={row['rank']} reps={row['input_repeats']} decay={row['decay_rate']:.3f} "
            f"rest={row['resting_mp']:.2f} thresh={row['spike_threshold']:.2f} "
            f"rec={row['recurrent_scale']:.3f} input={row['input_gain']:.3f} tau={row['spike_tau']:.2f} "
            f"vscale={row['feature_voltage_scale']:.2f} winit={row['weight_init_scale']:.2e} | "
            f"gamma={row['gamma']:.3f} actor_lr={row['actor_lr']:.2e} critic_lr={row['critic_lr']:.2e} "
            f"reward_scale={row['reward_scale']:.3g} entropy={row['entropy_beta']:.2e} batch={row['batch_episodes']} "
            f"grad_clip={row['max_grad_norm']:.2f} policy_step={row['max_policy_logit_step']:.2f} "
            f"entropy_floor={row['entropy_floor']:.2f} ppo_clip={row['ppo_clip']:.2f} ppo_epochs={row['ppo_epochs']} "
            f"target_kl={row['target_kl']:.3f} backtracks={row['line_search_backtracks']} "
            f"hard={row['hard_start_prob']:.2f} fail_w={row['failure_weight_bonus']:.1f}/{row['failure_window']} "
            f"buf={row['failure_buffer_size']} starts={row['hard_starts']} seed={row['seed']}"
        )
    return rows


def apply_optimized_result(result, train_episodes: int | None = None):
    global RESERVOIR_CONFIG, RL_CONFIG
    RESERVOIR_CONFIG = result["reservoir_config"]
    RL_CONFIG = result["rl_config"] if train_episodes is None else replace(result["rl_config"], train_episodes=int(train_episodes))
    return RESERVOIR_CONFIG, RL_CONFIG


apply_search_result = apply_optimized_result

# Example usage. Start small, then increase n_trials/train_episodes once the engine behavior is sane.
study, optimization_results = run_spike_engine_optuna_search(n_trials=20, train_episodes=300, eval_episodes=20, seed=123)
leaderboard = print_spike_engine_optimizer_leaderboard(optimization_results, top_k=10)
RESERVOIR_CONFIG, RL_CONFIG = apply_optimized_result(optimization_results[0], train_episodes=1500)
env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)


In [ ]:
# Optional after running the Optuna cell above:
history = {"returns": [], "eval_history": []}
RESERVOIR_CONFIG, RL_CONFIG = apply_optimized_result(optimization_results[0], train_episodes=1500)
env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)


## Checkpoint Helpers

Small helpers for saving and reloading the trained actor-critic readout and the configs used to create it.

In [ ]:
def save_cartpole_checkpoint(path: str, reservoir_config=RESERVOIR_CONFIG, rl_config=RL_CONFIG, agent=agent, history=history):
    failure_buffer = np.asarray(getattr(agent, "failure_state_buffer", []), dtype=np.float32)
    payload = {
        "reservoir_config": reservoir_config.__dict__,
        "rl_config": rl_config.__dict__,
        "actor_W": agent.actor_W,
        "value_W": agent.value_W,
        "updates": np.asarray([agent.updates], dtype=np.int64),
        "returns": np.asarray(history.get("returns", []), dtype=np.float32),
        "failure_state_buffer": failure_buffer,
    }
    np.savez_compressed(path, **payload)
    print(f"saved {path}")


def load_cartpole_checkpoint(path: str):
    data = np.load(path, allow_pickle=True)
    reservoir_config = SpikeEngineReservoirConfig(**data["reservoir_config"].item())
    rl_config = RLConfig(**data["rl_config"].item())
    env, reservoir, agent = make_system(reservoir_config, rl_config)
    agent.actor_W = data["actor_W"].astype(np.float32)
    agent.value_W = data["value_W"].astype(np.float32)
    agent.updates = int(data["updates"][0])
    if "failure_state_buffer" in data:
        agent.failure_state_buffer = [row.astype(np.float32) for row in data["failure_state_buffer"]]
        agent.last_failure_buffer_size = len(agent.failure_state_buffer)
    history = {"returns": data["returns"].astype(np.float32), "eval_history": []}
    return env, reservoir, agent, reservoir_config, rl_config, history

# save_cartpole_checkpoint("cartpole_spike_engine_checkpoint.npz")
# env, reservoir, agent, RESERVOIR_CONFIG, RL_CONFIG, history = load_cartpole_checkpoint("cartpole_spike_engine_checkpoint.npz")


## Tuning Notes

- If activity dies out, increase `input_gain` or `recurrent_scale` slightly.
- If every neuron saturates, reduce `input_gain`, increase `decay_rate`, or lower `recurrent_scale`.
- This notebook reads features back to CPU every environment step. The next production step is a GPU feature sampler and a batched multi-engine rollout path.